## Dependencies


In [1]:
import os
import time
import json
import torch
import functools
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import torch.nn as nn
import torch.optim as optim
from src.music import util
from src.music import generation
import matplotlib.pyplot as plt
from scipy.io.wavfile import write
from collections import defaultdict
from IPython import display as ipythondisplay


## Dataset


In [2]:
# Download the dataset
songs = generation.load_training_data()

# Print one of the songs to inspect it in greater detail!
example_song = songs[0]
print("\nExample song: ")
print(example_song)

Found 817 songs in text

Example song: 
X:1
T:Alexander's
Z: id:dc-hornpipe-1
M:C|
L:1/8
K:D Major
(3ABc|dAFA DFAd|fdcd FAdf|gfge fefd|(3efe (3dcB A2 (3ABc|!
dAFA DFAd|fdcd FAdf|gfge fefd|(3efe dc d2:|!
AG|FAdA FAdA|GBdB GBdB|Acec Acec|dfaf gecA|!
FAdA FAdA|GBdB GBdB|Aceg fefd|(3efe dc d2:|!


In [ ]:
# Convert the ABC notation to audio file and listen to it
generation.play_song(example_song)

In [3]:
# single string containing all songs
songs_joined = "\n\n".join(songs)

# Find all unique characters
vocab = sorted(set(songs_joined))
print("There are", len(vocab), "unique characters in the dataset")

There are 83 unique characters in the dataset


## Process the dataset for the learning task



### Vectorize the text


##### Define numerical representation of text

In [4]:
char2idx = {u: i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

print('{')
for char, _ in zip(char2idx, range(20)):
    print('  {:4s}: {:3d},'.format(repr(char), char2idx[char]))
print('  ...\n}')

{
  '\n':   0,
  ' ' :   1,
  '!' :   2,
  '"' :   3,
  '#' :   4,
  "'" :   5,
  '(' :   6,
  ')' :   7,
  ',' :   8,
  '-' :   9,
  '.' :  10,
  '/' :  11,
  '0' :  12,
  '1' :  13,
  '2' :  14,
  '3' :  15,
  '4' :  16,
  '5' :  17,
  '6' :  18,
  '7' :  19,
  ...
}


##### Vectorize the songs string

In [5]:

def vectorize_string(string):
  return np.array([char2idx[char] for char in string])

vectorized_songs = vectorize_string(songs_joined)

print ('{} --------> {}'.format(repr(songs_joined[:10]), vectorized_songs[:10]))
assert isinstance(vectorized_songs, np.ndarray), "returned result should be a numpy array"
print("length of the vectorized songs string:", vectorized_songs.shape[0])

'X:1\nT:Alex' --------> [49 22 13  0 45 22 26 67 60 79]
length of the vectorized songs string: 200679


### Create training examples and targets

In [6]:
def get_batch(vectorized_songs, seq_length, batch_size):
    n = vectorized_songs.shape[0] - 1
    # randomly choose the starting indices for the examples in the training batch
    idx = np.random.choice(n - seq_length, batch_size)

    # list of input sequences for the training batch
    input_batch = np.array([
        vectorized_songs[i: i + seq_length] for i in idx
    ])

    # list of output sequences for the training batch
    output_batch = np.array([
        vectorized_songs[i + 1: i + seq_length + 1] for i in idx
    ])

    # Convert the input and output batches to tensors
    x_batch = torch.tensor(input_batch, dtype=torch.long)
    y_batch = torch.tensor(output_batch, dtype=torch.long)

    return x_batch, y_batch

# Simple tests 
test_args = (vectorized_songs, 10, 2)
x_batch, y_batch = get_batch(*test_args)
assert x_batch.shape == (2, 10), "x_batch shape is incorrect"
assert y_batch.shape == (2, 10), "y_batch shape is incorrect"
print("Batch function works correctly!")


Batch function works correctly!


For each of these vectors, each index is processed at a single time step. So, for the input at time step 0, the model receives the index for the first character in the sequence, and tries to predict the index of the next character. At the next timestep, it does the same thing, but the RNN considers the information from the previous step, i.e., its updated state, in addition to the current input.

In [7]:
x_batch, y_batch = get_batch(vectorized_songs, seq_length=5, batch_size=1)

for i, (input_idx, target_idx) in enumerate(zip(x_batch[0], y_batch[0])):
    print("Step {:3d}".format(i))
    print("  input: {} ({:s})".format(input_idx, repr(idx2char[input_idx.item()])))
    print("  expected output: {} ({:s})".format(target_idx, repr(idx2char[target_idx.item()])))


Step   0
  input: 58 (np.str_('c'))
  expected output: 59 (np.str_('d'))
Step   1
  input: 59 (np.str_('d'))
  expected output: 82 (np.str_('|'))
Step   2
  input: 82 (np.str_('|'))
  expected output: 26 (np.str_('A'))
Step   3
  input: 26 (np.str_('A'))
  expected output: 31 (np.str_('F'))
Step   4
  input: 31 (np.str_('F'))
  expected output: 29 (np.str_('D'))


# model


* [`nn.Embedding`](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html): This is the input layer, consisting of a trainable lookup table that maps the numbers of each character to a vector with `embedding_dim` dimensions.
* [`nn.LSTM`](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html): Our LSTM network, with size `hidden_size`.
* [`nn.Linear`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html): The output layer, with `vocab_size` outputs.


## Define the RNN model

In [8]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size

        # Layer 1: Embedding layer to transform indices into dense vectors of a fixed embedding size
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # Layer 2: LSTM with hidden_size
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True) # TODO

        # Layer 3: Linear (fully-connected) layer that transforms the LSTM output into the vocabulary size
        self.fc = nn.Linear(hidden_size, vocab_size) # TODO

    def init_hidden(self, batch_size, device):
        # Initialize hidden state and cell state with zeros
        return (torch.zeros(1, batch_size, self.hidden_size).to(device),
                torch.zeros(1, batch_size, self.hidden_size).to(device))

    def forward(self, x, state=None, return_state=False):
        x = self.embedding(x)

        if state is None:
            state = self.init_hidden(x.size(0), x.device)
        out, state = self.lstm(x, state)

        out = self.fc(out)
        return out if not return_state else (out, state)



##### Let's instantiate the model!

In [9]:
vocab_size = len(vocab)
embedding_dim = 256
hidden_size = 1024
batch_size = 8

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

model = LSTMModel(vocab_size, embedding_dim, hidden_size).to(device)

# print out a summary of the model
print(model)

LSTMModel(
  (embedding): Embedding(83, 256)
  (lstm): LSTM(256, 1024, batch_first=True)
  (fc): Linear(in_features=1024, out_features=83, bias=True)
)


### Test out the RNN model

It's always a good idea to run a few simple checks on our model to see that it behaves as expected!

In [10]:
x, y = get_batch(vectorized_songs, seq_length=100, batch_size=32)
x = x.to(device)
y = y.to(device)


pred = model(x)
print("Input shape:      ", x.shape, "    # (batch_size, sequence_length)")
print("Prediction shape: ", pred.shape, "# (batch_size, sequence_length, vocab_size)")

Input shape:       torch.Size([32, 100])     # (batch_size, sequence_length)
Prediction shape:  torch.Size([32, 100, 83]) # (batch_size, sequence_length, vocab_size)


### Predictions from the untrained model

Sample the next token from the model's probability distribution. Unlike argmax, sampling allows varied generations instead of always choosing the single most likely token.

In [11]:
sampled_indices = torch.multinomial(torch.softmax(pred[0], dim=-1), num_samples=1)
sampled_indices = sampled_indices.squeeze(-1).cpu().numpy()
sampled_indices

array([58, 53, 50, 59, 58, 71,  7, 66, 46, 18, 56, 78, 60, 30, 27, 16, 52,
       73, 62, 53, 32, 57, 77, 48, 72,  8, 52,  1, 74, 58,  2, 36, 57, 21,
       72, 13, 35, 68, 62,  6, 47, 40, 46, 16, 37, 49,  8,  3, 36, 17, 12,
       59, 74, 44, 31, 17, 35, 76, 64, 67, 38, 47, 79, 18, 37, 38, 26, 82,
       18, 22, 39, 22, 23, 13, 71, 44, 70, 33, 57, 74, 43, 17, 73, 80, 27,
       65, 67, 49, 76, 46, 25, 64, 62, 36, 44, 82, 39, 15, 81, 54])

We can now decode these to see the text predicted by the untrained model:

In [ ]:
print("Input: \n", repr("".join(idx2char[x[0].cpu()])))
print()
print("Next Char Predictions: \n", repr("".join(idx2char[sampled_indices])))

## Training the model: loss and training operations

### Defining the loss function

In [12]:


cross_entropy = nn.CrossEntropyLoss() # instantiates the function

def compute_loss(labels, logits):
    """
    Inputs:
      labels: (batch_size, sequence_length)
      logits: (batch_size, sequence_length, vocab_size)

    Output:
      loss: scalar cross entropy loss over the batch and sequence length
    """

    # Batch the labels so that the shape of the labels should be (B * S,)
    batched_labels = labels.view(-1)
    
    # Batch the logits so that the shape of the logits should be (B * S, V)
    batched_logits = logits.view(-1, logits.size(-1))

    loss = cross_entropy(batched_logits, batched_labels)
    
    return loss

##### compute the loss on the predictions from the untrained model

In [13]:
y.shape  # (batch_size, sequence_length)
pred.shape  # (batch_size, sequence_length, vocab_size)

example_batch_loss = compute_loss(y, pred) 

print(f"Prediction shape: {pred.shape} # (batch_size, sequence_length, vocab_size)")
print(f"scalar_loss:      {example_batch_loss.mean().item()}")

Prediction shape: torch.Size([32, 100, 83]) # (batch_size, sequence_length, vocab_size)
scalar_loss:      4.418760299682617


### Hyperparameter setting and optimization

In [14]:

vocab_size = len(vocab)

params = dict(
    num_training_iterations = 3000,
    batch_size = 128,  # Experimented between 1 and 64
    seq_length = 100,  # Experimented between 50 and 500
    learning_rate = 1e-3,  # Experimented between 1e-5 and 1e-1
    embedding_dim = 256,
    hidden_size = 512,  # Experimented between 1 and 2048
)

with open("params_list.json") as f:
    params_list = json.load(f)

#### device currently used to train


In [15]:
print(device)
print(params_list)

mps
[]


#### split data and define training step / validation step

In [16]:
### Define optimizer and training operation ###
# 90% training, 10% validation
split_idx = int(len(vectorized_songs) * 0.9)

train_data = vectorized_songs[:split_idx]
val_data = vectorized_songs[split_idx:]

print("Training data:", len(train_data))
print("Validation data:", len(val_data))



def train_step(x, y):
    # Set the model's mode to train
    model.train()

    # Zero gradients for every step
    optimizer.zero_grad()

    # Forward pass
    y_hat = model(x)

    # Compute the loss
    '''TODO: compute the loss!'''
    loss = compute_loss(y, y_hat)

    # Backward pass
    # Backpropagate the loss
    # Update the model parameters using the optimizer
    loss.backward()
    optimizer.step()

    return loss

def validation_step(x, y):
    model.eval()

    with torch.no_grad():
        y_hat = model(x)
        loss = compute_loss(y, y_hat)

    return loss



Training data: 180611
Validation data: 20068


In [17]:
models_dir = Path.cwd() / "models"
models_dir.mkdir(exist_ok=True)

for params in params_list: 
    print(pd.Series(params))
    # Create a folder at the start of each run
    run_name = (
    f"lr_{params['learning_rate']:.0e}"
    f"_hs_{params['hidden_size']}"
    f"_bs_{params['batch_size']}"
    f"_seq_{params['seq_length']}"
    )
    run_dir = models_dir / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    # Save the parameters
    (run_dir / "params.json").write_text(json.dumps(params, indent=2))

    # instantiate a new LSTMModel model for training using the hyperparameters created above
    model = LSTMModel(vocab_size, params["embedding_dim"], params["hidden_size"])

    # Move the model to the device
    model.to(device)

    # instantiate an optimizer with its learning rate.
    optimizer = optim.Adam(model.parameters(), lr=params["learning_rate"])
    
    val_batches = []

    for _ in range(20):
        x_val, y_val = get_batch(val_data, params["seq_length"], params["batch_size"])

        x_val = torch.as_tensor(x_val, dtype=torch.long, device=device)
        y_val = torch.as_tensor(y_val, dtype=torch.long, device=device)

        val_batches.append((x_val, y_val))

    history = []
    val_history = []

    best_val_loss = float("inf")

    plotter = util.PeriodicPlotter(sec=2, xlabel='Iterations', ylabel='Loss')

    if hasattr(tqdm, '_instances'): tqdm._instances.clear() # clear if it exists
    for iter in tqdm(range(params["num_training_iterations"] + 1)):

        # Grab a batch and propagate it through the network
        x_batch, y_batch = get_batch(train_data, params["seq_length"], params["batch_size"])

        # Convert numpy arrays to PyTorch tensors
        x_batch = torch.tensor(x_batch, dtype=torch.long).to(device)
        y_batch = torch.tensor(y_batch, dtype=torch.long).to(device)

        # Take a train step
        loss = train_step(x_batch, y_batch)

        # Update the progress bar and visualize within notebook
        history.append(loss.item())
        plotter.plot_train_val(history, val_history, val_interval=100)

        # Save model checkpoint
        if iter % 100 == 0:
            # 1. Validation
            val_losses = []

            for x_val, y_val in val_batches:
                val_loss = validation_step(x_val, y_val)
                val_losses.append(val_loss.item())

            avg_val_loss = np.mean(val_losses)
            val_history.append(avg_val_loss)

            # 2. Print progress
            print(
                f"\nIteration: {iter + 1} | "
                f"Train Loss: {loss.item():.4f} | "
                f"Validation Loss: {avg_val_loss:.4f}"
            )


            # 3. Save checkpoint only if validation improves
            if avg_val_loss < best_val_loss:

                best_val_loss = avg_val_loss

                torch.save(
                    {
                        "model_state_dict": model.state_dict(),
                        "params": params,
                        "val_loss": best_val_loss,
                    },
                    run_dir / "best_model.pt",
                )

                print("Best model saved!")

    # 4. Update plot
    save_path = run_dir / "loss.png"
    save_path.parent.mkdir(parents=True, exist_ok=True)

    plotter.plot_train_val(
        history,
        val_history,
        val_interval=100,
        save_path=save_path
    )

In [18]:
results = defaultdict(list)

for run_dir in sorted(models_dir.iterdir()):
    if not run_dir.is_dir():
        print(run_dir, "not found")
        continue

    pt_path = run_dir / "best_model.pt"
    if not pt_path.exists():
        continue

    checkpoint = torch.load(pt_path, map_location=device, weights_only=False)

    checkpoint |= checkpoint.pop("params")
    for key, value in checkpoint.items():
        results[key].append(value)

    

sorted_models = pd.DataFrame(results).sort_values(by="val_loss").reset_index(drop=True)


print(sorted_models.drop(columns=["model_state_dict"]).to_string())
    

    val_loss  num_training_iterations  batch_size  seq_length  learning_rate  embedding_dim  hidden_size
0   1.368786                     3000         128         100         0.0005            256          256
1   1.371095                     3000         128         100         0.0001            256          256
2   1.373872                     3000         128         100         0.0010            256          256
3   1.378657                     3000         128         100         0.0010            256          512
4   1.383801                     3000         128         100         0.0010            256         1024
5   1.402358                     3000         128         100         0.0005            256          128
6   1.406509                     3000         128         100         0.0010            256          128
7   1.413317                     3000         128         100         0.0050            256          256
8   1.415073                     3000         128      

## Generate music using the RNN model



### The prediction procedure



![LSTM inference](https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/master/lab1/img/lstm_inference.png)


In [19]:
best_model = sorted_models.iloc[1]
print(best_model["val_loss"])
model = LSTMModel(
    vocab_size=vocab_size, 
    embedding_dim=int(best_model["embedding_dim"]), 
    hidden_size=int(best_model["hidden_size"])
).to(device)

model.load_state_dict(best_model["model_state_dict"], strict=True)
model.eval()


1.3710948705673218


LSTMModel(
  (embedding): Embedding(83, 256)
  (lstm): LSTM(256, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=83, bias=True)
)

In [20]:
def generate_text(model, start_string, generation_length=1000):
    input_ids = [char2idx[char] for char in start_string]
    input_idx = torch.tensor([input_ids], dtype=torch.long, device=device)

    model.eval()
    state = model.init_hidden(input_idx.size(0), device)
    text_generated = []

    with torch.no_grad():
        for _ in tqdm(range(generation_length)):
            predictions, state = model(
                input_idx,
                state=state,
                return_state=True,
            )

            # Use logits for the final position: shape becomes [1, vocab_size]
            next_logits = predictions[:, -1, :]
            probabilities = torch.softmax(next_logits, dim=-1)
            input_idx = torch.multinomial(probabilities, num_samples=1)

            # Convert the single sampled token to a Python int for lookup
            text_generated.append(idx2char[input_idx.item()])

    return start_string + "".join(text_generated)

In [21]:
start_string = """X:1
T:Generated
M:2/4
L:1/8
K:G
"""
generated_text = generate_text(model, start_string)

100%|██████████| 1000/1000 [00:00<00:00, 1633.17it/s]


### Play back the generated music!

In [ ]:
generated_songs = generation.extract_song_snippet(generated_text)
for i, song in enumerate(generated_songs, start=1):
    print(f"\n--- Generated song {i} ---")
    for line_num, line in enumerate(song.splitlines(), start=1):
        print(f"{line_num:2}: {line}")

for i, song in enumerate(generated_songs):
  # Synthesize the waveform from a song
  waveform = generation.play_song(song)

  # If its a valid song (correct syntax), lets play it!
  if waveform:
    print("Generated song", i)
    ipythondisplay.display(waveform)

    numeric_data = np.frombuffer(waveform.data, dtype=np.int16)
    wav_file_path =  f"output/output_{i}.wav"
    write(wav_file_path, 88200, numeric_data)

    # save your song to the Comet interface -- you can access it there
    # experiment.log_asset(wav_file_path)